# 01 · Search and train the decision tree (one dataset per run)

1. Loads the selected dataset profile (leakage-aware column removal, numeric features only).
2. For every K in {20, 25, 30, 35, 40}: **Silverback** (global search) → **Dolphin** (local refinement) → **pruning** search, each scored by cross-validation inside a leakage-safe `Pipeline`.
3. Selects the model by **validation MCC** only, refits on train + validation and evaluates the **held-out test split once**.
4. Saves the fitted pipeline (`.pkl`), `metadata_*.json`, `validation_summary_*.csv` and `search_history_*.csv` (convergence figure).
5. Repeats the selected configuration on independent splits (mean ± std).

**Outputs**: `results/esp32_dt_final_results/<dataset>/`, `results/article_results/<dataset>/`, `results/figures/<dataset>/`

**Runtime** depends on `PAPER_MODE` (see `src/run_settings.py`): `smoke` takes minutes, `quick` is about 10× cheaper than `full`, `full` uses the paper settings.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from framework import *                      # config, optimisers, pipeline builders, main()
from framework import _json_default
import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("Dataset:", ACTIVE_DATASET, "| CSV:", DEFAULT_DATASET_PATH)
assert Path(DEFAULT_DATASET_PATH).exists(), f"CSV not found: {DEFAULT_DATASET_PATH}  ->  put it in <package>/data/"

## Run the optimiser

In [ ]:
final_pipeline, label_encoder, metadata, validation_results = main(MAIN_OVERRIDES)
print("\n✨ Optimisation complete!  best K =", metadata["best_k"])

## Validation K-sweep and search convergence

In [ ]:
d = Path("esp32_dt_final_results") / ACTIVE_DATASET
val = pd.read_csv(sorted(d.glob("validation_summary_*.csv"))[-1])
display(val)

# --- Figure: validation MCC / macro-F1 vs K, with model-size proxy ---------------------
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(val["k"], val["validation_mcc"], "o-", label="validation MCC")
ax.plot(val["k"], val["validation_macro_f1"], "s--", label="validation macro-F1")
ax.set_xlabel("selected features K"); ax.set_ylabel("score"); ax.legend(loc="lower right")
ax2 = ax.twinx(); ax2.bar(val["k"], val["model_size_kb_proxy"], width=1.6, alpha=.18, color="gray")
ax2.set_ylabel("size proxy (KB)"); ax2.grid(False)
ax.set_title(f"{ACTIVE_DATASET}: validation score vs K  (selected K = {metadata['best_k']})")
fig.savefig(FIGURES_DIR / "validation_k_sweep.png", dpi=300, bbox_inches="tight"); plt.show()

# --- Figure: best-so-far fitness across Silverback -> Dolphin -> Pruning --------------
hist_files = sorted(d.glob("search_history_*.csv"))
if hist_files:
    h = pd.read_csv(hist_files[-1])
    hk = h[h["k"] == metadata["best_k"]].copy()
    order = {"silverback": 0, "dolphin": 1, "prune": 2}
    hk["o"] = hk["stage"].map(order)
    hk = hk.sort_values(["o", "eval_in_stage"]).reset_index(drop=True)
    hk["eval"] = np.arange(1, len(hk) + 1)
    hk["best_so_far"] = hk["fitness"].cummax()
    fig, ax = plt.subplots(figsize=(7, 3.8))
    colors = {"silverback": "#4C72B0", "dolphin": "#55A868", "prune": "#C44E52"}
    for st, g in hk.groupby("stage", sort=False):
        ax.scatter(g["eval"], g["fitness"], s=14, alpha=.55, color=colors[st], label=f"{st} evaluations")
    ax.step(hk["eval"], hk["best_so_far"], where="post", color="black", lw=1.6, label="best so far")
    for st in ("dolphin", "prune"):
        first = hk.loc[hk["stage"] == st, "eval"].min()
        if pd.notna(first): ax.axvline(first - .5, color="gray", ls=":", lw=1)
    ax.set_xlabel("cumulative fitness evaluations"); ax.set_ylabel("fitness (CV)")
    ax.set_title(f"{ACTIVE_DATASET}: Silverback → Dolphin → Pruning  (K = {metadata['best_k']})")
    ax.legend(fontsize=7, loc="lower right")
    fig.savefig(FIGURES_DIR / "optimizer_convergence.png", dpi=300, bbox_inches="tight"); plt.show()
else:
    print("No search_history_*.csv found (it is written by this notebook's main() run).")

## Multi-seed robustness (mean ± std over independent splits)
The selected `(K, params)` is **re-trained and re-tested on several different random splits**. This measures split-to-split variability of the *final model*, not a new search.

In [ ]:
# =============================================================================
# MULTI-SEED ROBUSTNESS CHECK
# =============================================================================
N_REPEAT_SEEDS = N_REPEAT_SEEDS_CFG
REPEAT_SEEDS = [SEED + 100 + i for i in range(N_REPEAT_SEEDS)]  # distinct from SEED itself

print(f"Dataset profile : {ACTIVE_DATASET!r}")
print(f"Reusing search result: best_k={metadata['best_k']}, best_params={metadata['best_params']}")
print(f"Running {N_REPEAT_SEEDS} independent seeds: {REPEAT_SEEDS}\n")

# Reload once; only the split random_state changes per seed below.
X_ms, y_ms, le_ms, numeric_cols_ms, _shape_ms = load_and_prepare_dataframe(DEFAULT_DATASET_PATH)

repeat_rows = []
for seed_i in REPEAT_SEEDS:
    X_train_val_i, X_test_i, y_train_val_i, y_test_i = train_test_split(
        X_ms, y_ms, test_size=TEST_SIZE, stratify=y_ms, random_state=seed_i
    )
    X_train_i, X_val_i, y_train_i, y_val_i = train_test_split(
        X_train_val_i, y_train_val_i,
        test_size=VAL_SIZE, stratify=y_train_val_i, random_state=seed_i
    )
    X_final_i = pd.concat([X_train_i, X_val_i], axis=0)
    y_final_i = np.concatenate([y_train_i, y_val_i])

    rare_i, extremely_rare_i = infer_rare_classes(y_train_i)

    pipe_i = build_pipeline(metadata["best_params"], metadata["best_k"], seed_i)
    pipe_i.fit(X_final_i, y_final_i)
    metrics_i = evaluate_on_split(pipe_i, X_test_i, y_test_i, rare_i, extremely_rare_i)

    row = {"seed": seed_i}
    row.update({k: v for k, v in metrics_i.items() if isinstance(v, (int, float, bool))})
    repeat_rows.append(row)

    print(f"  seed {seed_i}: accuracy={metrics_i['accuracy']:.4f}  "
          f"macro_f1={metrics_i['macro_f1']:.4f}  mcc={metrics_i['mcc']:.4f}  "
          f"minority_recall={metrics_i.get('minority_recall', float('nan')):.4f}")
    gc.collect()

repeats_df = pd.DataFrame(repeat_rows)

METRIC_COLS_FOR_SUMMARY = [c for c in [
    "accuracy", "macro_f1", "weighted_f1", "mcc", "kappa",
    "small_class_f1", "minority_recall", "rare_class_mcc",
    "extremely_rare_mcc", "embedded_size_kb", "latency_ms_per_sample",
] if c in repeats_df.columns]

summary_stats = repeats_df[METRIC_COLS_FOR_SUMMARY].agg(["mean", "std"]).T
summary_stats.columns = ["mean", "std"]

print(f"\nMulti-seed test-set summary (mean ± std over {N_REPEAT_SEEDS} seeds):")
for m in METRIC_COLS_FOR_SUMMARY:
    print(f"  {m:22s}: {summary_stats.loc[m, 'mean']:.4f} ± {summary_stats.loc[m, 'std']:.4f}")

MULTISEED_DIR = Path("article_results") / ACTIVE_DATASET
MULTISEED_DIR.mkdir(parents=True, exist_ok=True)
repeats_df.to_csv(MULTISEED_DIR / "multi_seed_repeats_raw.csv", index=False)
summary_stats.to_csv(MULTISEED_DIR / "multi_seed_repeats_summary.csv")

# Fold mean/std into article_results_summary.json if that file already
# exists for this run (safe no-op otherwise -- run the Article evidence
# package cell before or after this one, order doesn't matter).
summary_json_path = MULTISEED_DIR / "article_results_summary.json"
if summary_json_path.exists():
    with open(summary_json_path, "r", encoding="utf-8") as f:
        existing_summary = json.load(f)
    existing_summary["multi_seed_n"] = N_REPEAT_SEEDS
    existing_summary["multi_seed_test_metrics_mean_std"] = {
        m: {"mean": float(summary_stats.loc[m, "mean"]), "std": float(summary_stats.loc[m, "std"])}
        for m in METRIC_COLS_FOR_SUMMARY
    }
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(existing_summary, f, indent=2, default=str)
    print(f"\nUpdated {summary_json_path} with multi-seed mean/std.")
else:
    print(f"\nNote: {summary_json_path} does not exist yet. Run the Article evidence "
          "package cell (below) too -- multi-seed stats are saved either way in "
          "multi_seed_repeats_summary.csv, and will be folded in automatically once "
          "that file exists.")

print(f"\nSaved: {MULTISEED_DIR / 'multi_seed_repeats_raw.csv'}")
print(f"Saved: {MULTISEED_DIR / 'multi_seed_repeats_summary.csv'}")

In [ ]:
# =============================================================================
# MULTI-SEED COMPARISON FIGURE
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, metric in zip(axes, ["mcc", "macro_f1"]):
    if metric not in summary_stats.index:
        ax.axis("off")
        continue
    mean_val = summary_stats.loc[metric, "mean"]
    std_val = summary_stats.loc[metric, "std"]
    ax.bar([ACTIVE_DATASET], [mean_val], yerr=[std_val], capsize=8, color="#4C72B0")
    ax.set_title(f"Test {metric.upper()} — mean ± std ({N_REPEAT_SEEDS} seeds)")
    ax.set_ylim(0, 1)
    ax.text(0, mean_val + std_val + 0.02, f"{mean_val:.4f} ± {std_val:.4f}",
            ha="center", fontsize=10)

fig.tight_layout()
fig_path = FIGURES_DIR / "multi_seed_robustness.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

**Next:** `02_evaluation_baselines_ablation.ipynb` for the same dataset.